## Real Data Examples

In [99]:
import importlib
import subprocess

def install_if_missing(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        subprocess.run(["pip", "install", package], check=True)

In [ ]:
import os
import sys
import torch

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/My Drive/Colab Notebooks/WGF')

from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import argparse
import torch.nn as nn
import torch.optim as optim
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy.spatial.distance import cdist, pdist
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle
import traceback
install_if_missing("pyreadr")
import pyreadr
from wgf_minimal import sample_wgf_new
from wgf_minimal import impute_wgf_python

install_if_missing("hyperimpute")

from hyperimpute.plugins.imputers import Imputers
import scipy

#sys.path.append(str(Path("MIRI-Imputation").resolve()))
#from src.imputer_wrapper import impute_now

## Enable usage of R methods
os.environ['RENV_CONFIG_AUTOLOAD_ENABLED'] = 'FALSE'
os.environ['R_PROFILE_USER'] = ''
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

ro.r('''
install.packages("mice", repos="https://cloud.r-project.org")
install.packages("missForest", repos="https://cloud.r-project.org")
''')

(as ‘lib’ is unspecified)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).








	‘/tmp/RtmpimklEc/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpimklEc/downloaded_packages’



### Set parameters and load existing results if available

In [129]:
### SETTINGS ###

datasets = ["parkinsons"] #"gas", "pumadyn32nm", "scm1d", "scm20d",

methods = ['mice_cart', 'miri', 'wgf_new' ] # 'wgf_new', 'mice_cart', "mice_rf", "missForest", 'miri'

## Resampling params (still relevant)
param_vals = {
    "n_runs": 1,
    "init": "ColBT",
    "T": 100
}

In [130]:
def impute_bootstrap_per_col(X, M):
    n, d = X.shape
    for j in range(d):
        observed_mask = M[:, j] == 1
        missing_mask = M[:, j] == 0

        observed_values = X[observed_mask, j]
        if observed_values.numel() == 0:
            raise ValueError(f"Value of column {j} is always missing. Need to be observed at least once.")

        num_missing = missing_mask.sum()
        if num_missing > 0:
            rand_idx = torch.randint(
                0, observed_values.shape[0],
                (num_missing,),
                device=X.device
            )
            X[missing_mask, j] = observed_values[rand_idx]



def energy_distance(X, Y, scale):

    if scale:
        center = np.nanmean(X, axis=0)
        scale = np.nanstd(X, axis=0, ddof=1)

        X = (X - center) / scale

        # Scale imputed data using original data's mean and std
        Y = (Y - center) / scale

    XY = cdist(X, Y)
    XX = cdist(X, X)
    YY = cdist(Y, Y)
    return (2 * XY.mean() - XX.mean() - YY.mean()) * X.shape[0] / 2

def energy_distance_fixed_X(X, XX_mean, Y):
    XY_mean = cdist(X, Y).mean()
    n = len(Y)
    YY_mean = pdist(Y).mean() * (n - 1) / n
    return 2 * XY_mean - XX_mean - YY_mean

In [131]:
# Define the log file path
save_dir = 'results'
log_file = os.path.join(save_dir, 'parameter_log.csv')

# Check if the log file exists and read it
if os.path.exists(log_file):
    param_log = pd.read_csv(log_file)
else:
    param_log = pd.DataFrame(columns=['ID', 'dataset', 'method', 'init', 'T'])

# Determine the next ID
next_id = 1 if param_log.empty else param_log['ID'].max() + 1

# Initialize results storage
results = {"data": {}, "metrics": {}, "Xhat_store": {}}

for dataset in datasets:
    results["data"][dataset] = {}
    results["metrics"][dataset] = {}
    results["Xhat_store"][dataset] = {}
    
    for method in methods:
        # Check if this (dataset, method) combination already exists
        matching = param_log.query(
            f"dataset == {repr(dataset)} & method == {repr(method)} & "
            f"init == {repr(param_vals['init'])} & T == {repr(param_vals['T'])}"
        )

        if not matching.empty:
            print(f"[SKIP] {dataset} / {method} already run (ID={matching['ID'].iloc[0]})")
            # Optionally load existing result here
            continue

        print(f"[RUN]  {dataset} / {method}")

        # --- your method logic here ---
        # e.g., Xhat = run_method(method, X_obs, ...)

        # Log this combination
        new_row = {'ID': next_id, 'dataset': dataset, 'method': method, **param_vals}
        param_log = pd.concat([param_log, pd.DataFrame([new_row])], ignore_index=True)
        next_id += 1

# Save updated log
os.makedirs(save_dir, exist_ok=True)
param_log.to_csv(log_file, index=False)


##Continue here!!

[SKIP] parkinsons / mice_cart already run (ID=1)
[RUN]  parkinsons / miri
[SKIP] parkinsons / wgf_new already run (ID=2)


In [132]:
for dataset in datasets:
    for run in range(param_vals["n_runs"]):
        torch.manual_seed(run + param_vals["n_runs"])
        np.random.seed(run + param_vals["n_runs"])

        Xstar_df = pyreadr.read_r(f"{base_path}/data/datasets/split/test.{ratio}.1.{dataset}.RDS")[None]
        X_miss_df = pyreadr.read_r(f"{base_path}/results/amputedsplit/mar.{ratio}.1.{dataset}.RDS")[None]
        ## M=1-M in the paper
        M_np = (~X_miss_df.isna()).astype(int).values
        M_tensor = torch.tensor(M_np, dtype=torch.float32)
        X_miss_tensor = torch.tensor(X_miss_df.values, dtype=torch.float32)
        #X0_tensor = impute_bootstrap_per_col(X_miss_tensor, M_tensor)

        impute_bootstrap_per_col(X_miss_tensor, M_tensor)
        X0_np = X_miss_tensor.numpy()  # read the in-place modified tensor
        results["data"][dataset][run] = {
            "Xstar": Xstar_df,
            "X_miss": X_miss_df,
            "M": M_np,
            "X0": X0_np  # convert back to numpy after bootstrap
        }

### Run sample methods and compute metrics (only those that are not available yet) 

In [133]:
ratio = 0.2
base_path = "/content/drive/My Drive/Colab Notebooks/WGF"

for dataset in datasets:
    for method in methods:
        if method in results["metrics"][dataset] and len(results["metrics"][dataset][method]) > 0:
            print(f"{method} / {dataset} already done — skipping")
            continue

        print(f"Running {method} on {dataset}")
        results["metrics"][dataset][method] = []

        for run, run_data in results["data"][dataset].items():
            torch.manual_seed(1)
            np.random.seed(1)

            if method == "truth":
                Xhats = [sample_truth(param_vals["n_new"], param_vals["d"])]
            elif method == "miri":
                X0_tensor = torch.tensor(run_data["X0"], dtype=torch.float32)
                M_tensor = torch.tensor(run_data["M"], dtype=torch.float32)
                Xstar_tensor = torch.tensor(run_data["Xstar"].values, dtype=torch.float32)
                Xhats = [impute_now(X0_tensor, M_tensor, Xstar_tensor, "miri", max_rounds=15, batchsize=500, maxepochs=900, odesteps=100)[0]]
                #Xhats = [impute_now(run_data["X0"], run_data["M"], run_data["Xstar"], "miri", max_rounds=15, batchsize=500, maxepochs=900, odesteps=100)[0]]
            elif method == "wgf_new":
                #If we want to see all iterations!
                #X0_tensor = torch.tensor(run_data["X0"], dtype=torch.float64)
                #M_tensor = torch.tensor(run_data["M"], dtype=torch.float64)
                #Xhats = sample_wgf_new(X0_tensor, X0_tensor, M_tensor, T=param_vals["T"])

                Xhats = impute_wgf_python(run_data["X_miss"])
            elif method == "mice_cart" or method == "mice_rf" or method == "missForest":
                ro.r("set.seed(123)")

                mice = importr("mice")
                missForest = importr("missForest")

                X_df = run_data["X_miss"].copy()
                X_df.columns = [f"x{i}" for i in range(X_df.shape[1])]

                with localconverter(ro.default_converter + pandas2ri.converter):
                    r_df = ro.conversion.py2rpy(X_df)

                if method == "mice_cart":
                    res = mice.mice(r_df, method="cart", m=1, remove_collinear=False, eps=0)
                    complete = ro.r["complete"]
                    completed = complete(res, 1)
                    with localconverter(ro.default_converter + pandas2ri.converter):
                        completed_df = ro.conversion.rpy2py(completed)
                elif method == "mice_rf":
                    res = mice.mice(r_df, method="rf", m=1, remove_collinear=False, eps=0)
                    complete = ro.r["complete"]
                    completed = complete(res, 1)
                    with localconverter(ro.default_converter + pandas2ri.converter):
                        completed_df = ro.conversion.rpy2py(completed)
                elif method == "missForest":
                    res = missForest.missForest(r_df)
                    with localconverter(ro.default_converter + pandas2ri.converter):
                        completed_df = ro.conversion.rpy2py(res.rx2("ximp"))

                X_imputed = torch.tensor(completed_df.values, dtype=torch.float32)
                Xhats = [X_imputed]
            else:
                raise NotImplementedError("Method not implemented so far.")

            Xhats_df = pd.DataFrame(Xhats[0].detach().cpu().numpy() if isinstance(Xhats[0], torch.Tensor) else Xhats[0])
            pyreadr.write_rds(f"{base_path}/results/imputations{method}{ratio}.{dataset}.RDS", Xhats_df)

            Xstar_np = run_data["Xstar"].values

            if isinstance(Xhats, torch.Tensor):
                Xhats = [Xhats]

            for i, Xhat in enumerate(Xhats):
                Xid = f"{method}_{dataset}_{run}_{i}"
                results["Xhat_store"][dataset][Xid] = Xhat

                Xhat_np = Xhat.detach().cpu().numpy() if isinstance(Xhat, torch.Tensor) else (Xhat.values if isinstance(Xhat, pd.DataFrame) else np.array(Xhat))

                results["metrics"][dataset][method].append({
                    "run": run,
                    "iter": i,
                    "Xhat_id": Xid,
                    "energy": energy_distance(Xstar_np, Xhat_np, scale=True)
                })

Running mice_cart on parkinsons

 iter imp variable
  1   1  x0  x1  x2  x3  x4  x5
  2   1  x0  x1  x2  x3  x4  x5
  3   1  x0  x1  x2  x3  x4  x5
  4   1  x0  x1  x2  x3  x4  x5
  5   1  x0  x1  x2  x3  x4  x5


/usr/local/lib/python3.12/dist-packages/pyreadr/_pyreadr_writer.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['-3.056700006709434e-05' '-2.777699955913704e-05'
 '-6.107100034569157e-06' ... '2.436300019326154e-05'
 '-6.7071000557916705e-06' '8.852899554767646e-06']' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  pd_series.loc[pd.notnull(pd_series)] = pd_series.loc[pd.notnull(pd_series)].apply(lambda x: str(x))
/usr/local/lib/python3.12/dist-packages/pyreadr/_pyreadr_writer.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['-0.024096999317407608' '-0.03441699966788292' '0.041262999176979065' ...
 '0.05013300105929375' '-0.024646999314427376' '0.046783000230789185']' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  pd_series.l

Running miri on parkinsons


NameError: name 'impute_now' is not defined

In [128]:
results["metrics"]

{'parkinsons': {'mice_cart': [{'run': 0,
    'iter': 0,
    'Xhat_id': 'mice_cart_parkinsons_0_0',
    'energy': np.float64(2.2176372405774547)}],
  'wgf_new': [{'run': 0,
    'iter': 0,
    'Xhat_id': 'wgf_new_parkinsons_0_0',
    'energy': np.float64(3.8959440303484314)}]}}

### Save the results

In [ ]:
torch.save(results, results_file)

# Save the updated log file
param_log.to_csv(log_file, index=False)

In [ ]:
import os

log_file = 'results/parameter_log.csv'

if os.path.exists(log_file):
    os.remove(log_file)
    print("Deleted!")
else:
    print("File not found at that path")